# siebel_id_update_databricks

Incremental update logic for `REL_ID -> REL_ID_REGIE_KLANT` in Databricks.

This notebook assumes:
- A Databricks cluster is already attached.
- `spark` session is already available.
- Source tables already exist in Unity Catalog / Hive metastore.

What this notebook does:
- Reads source tables (`direct_bank`, `ggm_np`).
- Creates state/output tables if missing.
- Runs incremental mapping update.

In [ ]:
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

# Databricks runtime already provides SparkSession as `spark`.
assert isinstance(spark, SparkSession), "This notebook must run on Databricks with an active Spark session."

In [ ]:
# Runtime parameters
target_db = "shebang"
state_db = "shebang_state"

table_output = "output"

table_edge_snapshot = "edge_snapshot"
table_seed_snapshot = "seed_snapshot"
table_seed_membership = "seed_membership"
table_run_audit = "run_audit"

direct_bank_source_path = "abfss://siebel-idaa-cdf@edlcorestdeuprod0001.dfs.core.windows.net/cdf_ggm_np_drc_bnk_hist/1/data/"
ggm_np_source_path = "abfss://siebel-idaa-cdf@edlcorestdeuprod0001.dfs.core.windows.net/cdf_ggm_np_hist/2/data/"
seed_membership_bootstrap_path = "abfss://ckb@edlcorestdeuprod0001.dfs.core.windows.net/klant_treintjes/history/1/data/"

max_iter = 40
checkpoint_every = 2
open_ended_ts = "9999-12-31 00:00:00"
fail_on_incomplete = True
seed_tie_breaker = "latest_then_numeric"
show_samples = True

In [ ]:
from uuid import uuid4


def has_rows(df: DataFrame) -> bool:
    """Check whether a DataFrame contains at least one row."""
    print("[siebel_id_update] has_rows: checking if DataFrame has at least one row")
    return df.limit(1).count() > 0


def _create_table_with_retry(spark: SparkSession, full_table_name: str, df: DataFrame, fmt: str) -> None:
    """Create a managed table and recover from orphan managed locations when necessary."""
    print(f"[siebel_id_update] _create_table_with_retry: creating table {full_table_name} with format={fmt}")
    try:
        df.write.mode("overwrite").format(fmt).saveAsTable(full_table_name)
    except Exception as exc:
        if "LOCATION_ALREADY_EXISTS" not in str(exc):
            raise
        print(
            f"[siebel_id_update] _create_table_with_retry: LOCATION_ALREADY_EXISTS for {full_table_name}, dropping and retrying"
        )
        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
        df.write.mode("overwrite").format(fmt).saveAsTable(full_table_name)


def overwrite_table_in_place(spark: SparkSession, full_table_name: str, df: DataFrame, fmt: str = "delta") -> None:
    """Replace table contents without dropping the governed table object first."""
    print(f"[siebel_id_update] overwrite_table_in_place: start table={full_table_name}, format={fmt}")
    if not spark.catalog.tableExists(full_table_name):
        print(f"[siebel_id_update] overwrite_table_in_place: table {full_table_name} does not exist, creating")
        _create_table_with_retry(spark, full_table_name, df, fmt)
        spark.catalog.refreshTable(full_table_name)
        return

    existing_schema = spark.table(full_table_name).schema
    existing_columns = [field.name for field in existing_schema]
    missing_columns = [column for column in existing_columns if column not in df.columns]
    extra_columns = [column for column in df.columns if column not in existing_columns]
    if missing_columns or extra_columns:
        raise ValueError(
            f"Schema mismatch for {full_table_name}: missing={missing_columns}, extra={extra_columns}"
        )

    aligned_df = df.select(*(F.col(field.name).cast(field.dataType).alias(field.name) for field in existing_schema))
    aligned_df = aligned_df.localCheckpoint(eager=True)

    temp_view_name = f"__overwrite_{uuid4().hex}"
    aligned_df.createOrReplaceTempView(temp_view_name)
    try:
        print(f"[siebel_id_update] overwrite_table_in_place: inserting overwrite into {full_table_name}")
        spark.sql(f"INSERT OVERWRITE TABLE {full_table_name} SELECT * FROM {temp_view_name}")
    finally:
        spark.catalog.dropTempView(temp_view_name)
    spark.catalog.refreshTable(full_table_name)
    print(f"[siebel_id_update] overwrite_table_in_place: finished table={full_table_name}")


def mutate_table_with_in_place_overwrite(spark: SparkSession, full_table_name: str, transform_fn) -> None:
    """Apply a transformation function and overwrite the same table object in place."""
    print(f"[siebel_id_update] mutate_table_with_in_place_overwrite: table={full_table_name}")
    current_df = spark.table(full_table_name)
    next_df = transform_fn(current_df)
    overwrite_table_in_place(spark, full_table_name, next_df)


def overwrite_table_with_staging_swap(spark: SparkSession, full_table_name: str, df: DataFrame, fmt: str = "delta") -> None:
    """Backward-compatible alias for in-place table overwrite."""
    print(f"[siebel_id_update] overwrite_table_with_staging_swap: alias to in-place overwrite for {full_table_name}")
    overwrite_table_in_place(spark, full_table_name, df, fmt=fmt)


def mutate_table_with_overwrite_swap(spark: SparkSession, full_table_name: str, transform_fn) -> None:
    """Backward-compatible alias for in-place overwrite mutation."""
    print(f"[siebel_id_update] mutate_table_with_overwrite_swap: alias to in-place mutation for {full_table_name}")
    mutate_table_with_in_place_overwrite(spark, full_table_name, transform_fn)


def _checkpoint(df: DataFrame, enabled: bool) -> DataFrame:
    """Checkpoint DataFrame conditionally to reduce lineage depth."""
    print(f"[siebel_id_update] _checkpoint: enabled={enabled}")
    return df.checkpoint(eager=True) if enabled else df


def is_open_ended_ts(col_name: str, open_end_ts: str) -> F.Column:
    """Create a predicate matching open-ended timestamp values."""
    print(f"[siebel_id_update] is_open_ended_ts: building predicate for column={col_name}")
    return F.to_timestamp(F.col(col_name)) == F.to_timestamp(F.lit(open_end_ts))


def assert_single_active_seed_per_rel_id(direct_df: DataFrame, open_end_ts: str) -> None:
    """Diagnostic helper: detect rel_id values with multiple active direct_bank seed keys (not enforced)."""
    print("[siebel_id_update] assert_single_active_seed_per_rel_id: validating active seed uniqueness")
    violations = (
        direct_df.filter((F.col("drc_bnk_f") == "Y") & is_open_ended_ts("edl_valid_to_ts", open_end_ts))
        .groupBy("rel_id")
        .agg(F.countDistinct("id_key").alias("active_seed_key_count"))
        .filter(F.col("active_seed_key_count") > 1)
    )

    if has_rows(violations):
        sample = [
            (str(r["rel_id"]), int(r["active_seed_key_count"]))
            for r in violations.select("rel_id", "active_seed_key_count").orderBy("rel_id").limit(20).collect()
        ]
        print(
            "[siebel_id_update] assert_single_active_seed_per_rel_id: multiple active seed keys detected (diagnostic only). "
            f"Sample rel_id counts: {sample}"
        )


def anti_diff(cur_df: DataFrame, prev_df: DataFrame, key_cols: list[str]) -> tuple[DataFrame, DataFrame]:
    """Compute key-based additions and removals between snapshots."""
    print(f"[siebel_id_update] anti_diff: computing anti-diff for keys={key_cols}")
    cur_keys = cur_df.select(*key_cols).dropDuplicates(key_cols)
    prev_keys = prev_df.select(*key_cols).dropDuplicates(key_cols)
    added = cur_keys.join(prev_keys, key_cols, "left_anti")
    removed = prev_keys.join(cur_keys, key_cols, "left_anti")
    return added, removed

In [ ]:
def normalize_active_direct_bank(direct_df: DataFrame, open_end_ts: str) -> DataFrame:
    """Normalize direct_bank records into graph-ready edges and seed metadata.

    Functionality:
    - Casts keys to string and parses validity timestamps.
    - Removes records with null/blank graph keys.
    - Keeps historical rel_id <-> np_sbl_id connectivity for graph traversal.
    - Deduplicates by (rel_id, np_sbl_id) using latest valid_from.

    Input parameters:
    - direct_df (DataFrame): Source direct_bank table.
    - open_end_ts (str): Open-ended timestamp marker (used later for seed eligibility).

    Output:
    - DataFrame: Columns rel_id, id_key, drc_bnk_f, edl_valid_to_ts, edl_valid_from_ts.
    """
    print("[siebel_id_update] normalize_active_direct_bank: start")
    filtered = (
        direct_df.select(
            F.col("rel_id").cast("string").alias("rel_id_s"),
            F.col("np_sbl_id").cast("string").alias("np_sbl_id_s"),
            F.col("drc_bnk_f").cast("string").alias("drc_bnk_f"),
            F.to_timestamp(F.col("edl_valid_to_dts")).alias("edl_valid_to_ts"),
            F.to_timestamp(F.col("edl_valid_from_dts")).alias("edl_valid_from_ts"),
        )
        .filter(
            F.col("rel_id_s").isNotNull()
            & F.col("np_sbl_id_s").isNotNull()
            & (F.trim(F.col("rel_id_s")) != "")
            & (F.trim(F.col("np_sbl_id_s")) != "")
        )
    )

    w = Window.partitionBy("rel_id_s", "np_sbl_id_s").orderBy(
        F.col("edl_valid_from_ts").desc_nulls_last(),
    )

    result = (
        filtered.withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .select(
            F.col("rel_id_s").alias("rel_id"),
            F.col("np_sbl_id_s").alias("id_key"),
            F.col("drc_bnk_f"),
            F.col("edl_valid_to_ts"),
            F.col("edl_valid_from_ts"),
        )
    )
    print("[siebel_id_update] normalize_active_direct_bank: done")
    return result


def normalize_active_ggm_np(ggm_df: DataFrame, open_end_ts: str) -> DataFrame:
    """Normalize ggm_np records into historical connectivity edges.

    Functionality:
    - Keeps all rel_id <-> ikb_no pairs as historical graph edges.
    - Casts and filters invalid key values.
    - Deduplicates repeated key pairs.

    Input parameters:
    - ggm_df (DataFrame): Source ggm_np table.
    - open_end_ts (str): Unused parameter kept for interface compatibility.

    Output:
    - DataFrame: Columns rel_id and id_key.
    """
    print("[siebel_id_update] normalize_active_ggm_np: start")
    _ = open_end_ts
    result = (
        ggm_df.select(
            F.col("rel_id").cast("string").alias("rel_id_s"),
            F.col("ikb_no").cast("string").alias("ikb_no_s"),
        )
        .filter(
            F.col("rel_id_s").isNotNull()
            & F.col("ikb_no_s").isNotNull()
            & (F.trim(F.col("rel_id_s")) != "")
            & (F.trim(F.col("ikb_no_s")) != "")
        )
        .dropDuplicates(["rel_id_s", "ikb_no_s"])
        .select(F.col("rel_id_s").alias("rel_id"), F.col("ikb_no_s").alias("id_key"))
    )
    print("[siebel_id_update] normalize_active_ggm_np: done")
    return result

In [ ]:
def expand_rel_component(start_rel_df: DataFrame, edges_df: DataFrame, max_iter: int = 40, checkpoint_every: int = 2) -> tuple[DataFrame, bool]:
    """Expand impacted rel_id nodes via iterative graph traversal.

    Functionality:
    - Traverses rel_id <-> id_key graph from a starting impacted set.
    - Stops when frontier is exhausted or max iteration limit is reached.

    Input parameters:
    - start_rel_df (DataFrame): Initial rel_id frontier.
    - edges_df (DataFrame): Graph edges with rel_id and id_key.
    - max_iter (int): Maximum number of traversal iterations.
    - checkpoint_every (int): Checkpoint frequency in iterations.

    Output:
    - tuple[DataFrame, bool]: (visited_rel_ids, truncated_flag).
    """
    print(
        f"[siebel_id_update] expand_rel_component: start max_iter={max_iter}, checkpoint_every={checkpoint_every}"
    )
    frontier = start_rel_df.select("rel_id").dropDuplicates(["rel_id"])
    frontier = _checkpoint(frontier, checkpoint_every > 0)
    visited = frontier
    truncated = False

    for step in range(max_iter):
        via_id = frontier.join(edges_df, "rel_id", "inner").select("id_key").dropDuplicates(["id_key"])
        next_rel = via_id.join(edges_df, "id_key", "inner").select("rel_id").dropDuplicates(["rel_id"])

        new_rel = next_rel.join(visited, "rel_id", "left_anti")
        if not has_rows(new_rel):
            print(f"[siebel_id_update] expand_rel_component: converged at step={step}")
            break

        visited = visited.unionByName(new_rel).dropDuplicates(["rel_id"])
        frontier = new_rel

        if step == max_iter - 1:
            truncated = True

        if checkpoint_every > 0 and (step + 1) % checkpoint_every == 0:
            visited = _checkpoint(visited, True)
            frontier = _checkpoint(frontier, True)

    print(f"[siebel_id_update] expand_rel_component: done truncated={truncated}")
    return visited, truncated


def expand_seed_pairs(active_seeds_df: DataFrame, edges_df: DataFrame, max_iter: int = 40, checkpoint_every: int = 2) -> tuple[DataFrame, bool]:
    """Expand reachable (seed_rel_id, rel_id) pairs for active seeds.

    Functionality:
    - Traverses graph from each active seed through rel_id <-> id_key edges.
    - Accumulates all reachable seed-to-rel_id mappings.

    Input parameters:
    - active_seeds_df (DataFrame): Active seeds with column seed_rel_id.
    - edges_df (DataFrame): Graph edges with rel_id and id_key.
    - max_iter (int): Maximum number of traversal iterations.
    - checkpoint_every (int): Checkpoint frequency in iterations.

    Output:
    - tuple[DataFrame, bool]: (visited_seed_pairs, truncated_flag).
    """
    print(
        f"[siebel_id_update] expand_seed_pairs: start max_iter={max_iter}, checkpoint_every={checkpoint_every}"
    )
    frontier = active_seeds_df.select(F.col("seed_rel_id"), F.col("seed_rel_id").alias("rel_id"))
    frontier = _checkpoint(frontier, checkpoint_every > 0)
    visited = frontier
    truncated = False

    for step in range(max_iter):
        via_id = (
            frontier.alias("f")
            .join(edges_df.alias("e"), F.col("f.rel_id") == F.col("e.rel_id"), "inner")
            .select(F.col("f.seed_rel_id"), F.col("e.id_key"))
            .dropDuplicates(["seed_rel_id", "id_key"])
        )

        next_rel = (
            via_id.alias("i")
            .join(edges_df.alias("e"), "id_key", "inner")
            .select(F.col("i.seed_rel_id"), F.col("e.rel_id"))
            .dropDuplicates(["seed_rel_id", "rel_id"])
        )

        new_pairs = next_rel.join(visited, ["seed_rel_id", "rel_id"], "left_anti")
        if not has_rows(new_pairs):
            print(f"[siebel_id_update] expand_seed_pairs: converged at step={step}")
            break

        visited = visited.unionByName(new_pairs).dropDuplicates(["seed_rel_id", "rel_id"])
        frontier = new_pairs

        if step == max_iter - 1:
            truncated = True

        if checkpoint_every > 0 and (step + 1) % checkpoint_every == 0:
            visited = _checkpoint(visited, True)
            frontier = _checkpoint(frontier, True)

    print(f"[siebel_id_update] expand_seed_pairs: done truncated={truncated}")
    return visited, truncated

In [ ]:
def ensure_tables(
    spark: SparkSession,
    target_db: str,
    state_db: str,
    seed_membership_source_path: str,
) -> None:
    """Create state and output Delta tables if they do not exist."""
    print(
        f"[siebel_id_update] ensure_tables: ensuring databases/tables target_db={target_db}, state_db={state_db}"
    )
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {state_db}")
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {target_db}")

    spark.sql(
        f"""
    CREATE TABLE IF NOT EXISTS {state_db}.{table_edge_snapshot} (
      source_table STRING,
      rel_id STRING,
      id_key STRING
    ) USING DELTA
    """
    )

    spark.sql(
        f"""
    CREATE TABLE IF NOT EXISTS {state_db}.{table_seed_snapshot} (
      seed_rel_id STRING,
      seed_valid_from_dts TIMESTAMP
    ) USING DELTA
    """
    )

    spark.sql(
        f"""
    CREATE TABLE IF NOT EXISTS {state_db}.{table_seed_membership}
    USING DELTA AS
    SELECT
      rel_id_regie_klant,
      rel_id
    FROM (
      SELECT
        rel_id_regie_klant,
        rel_id,
        EDL_VALID_FROM_DTS,
        EDL_VALID_TO_DTS,
        ROW_NUMBER() OVER (
          PARTITION BY rel_id, rel_id_regie_klant
          ORDER BY EDL_VALID_FROM_DTS DESC
        ) AS rn
      FROM parquet.`{seed_membership_source_path}`
    )
    WHERE rn = 1
    ORDER BY 1, 2
    """
    )

    spark.sql(
        f"""
    CREATE TABLE IF NOT EXISTS {state_db}.{table_run_audit} (
      run_id STRING,
      run_ts TIMESTAMP,
      changed_edges BIGINT,
      added_seeds BIGINT,
      removed_seeds BIGINT,
      impacted_rel_ids BIGINT,
      output_rows BIGINT,
      rel_truncated BOOLEAN,
      seed_truncated BOOLEAN
    ) USING DELTA
    """
    )

    spark.sql(
        f"""
    CREATE TABLE IF NOT EXISTS {target_db}.{table_output} (
      REL_ID STRING,
      REL_ID_REGIE_KLANT STRING
    ) USING DELTA
    """
    )
    print("[siebel_id_update] ensure_tables: done")

In [ ]:
def run_incremental_update_databricks(
    spark: SparkSession,
    target_db: str,
    state_db: str,
    max_iter: int = 40,
    checkpoint_every: int = 2,
    open_ended_ts: str = "9999-12-31 00:00:00",
    fail_on_incomplete: bool = True,
    seed_tie_breaker: str = "latest_then_numeric",
    show_samples: bool = False,
    seed_membership_source_path: str = "abfss://ckb@edlcorestdeuprod0001.dfs.core.windows.net/klant_treintjes/history/1/data/",
    direct_bank_source_path: str = "abfss://siebel-idaa-cdf@edlcorestdeuprod0001.dfs.core.windows.net/cdf_ggm_np_drc_bnk_hist/1/data/",
    ggm_np_source_path: str = "abfss://siebel-idaa-cdf@edlcorestdeuprod0001.dfs.core.windows.net/cdf_ggm_np_hist/2/data/",
) -> dict:
    """Execute the incremental mapping update in Databricks runtime."""
    print(
        f"[siebel_id_update] run_incremental_update_databricks: start target_db={target_db}, state_db={state_db}, max_iter={max_iter}"
    )
    if seed_tie_breaker not in {"numeric_then_lex", "latest_then_numeric"}:
        raise ValueError("seed_tie_breaker must be 'numeric_then_lex' or 'latest_then_numeric'")

    debug_suffix = uuid4().hex
    debug_table = f"{state_db}.seed_pairs_debug_{debug_suffix}"
    print(f"[siebel_id_update] debug table for this run: {debug_table}")

    if checkpoint_every > 0:
        print("[siebel_id_update] stage: configure checkpoint directory")
        spark.sparkContext.setCheckpointDir("/tmp/siebel_id_update_checkpoints")

    print("[siebel_id_update] stage: ensure state/output tables")
    ensure_tables(spark, target_db, state_db, seed_membership_source_path)

    print("[siebel_id_update] stage: read source data")
    direct_raw = spark.read.format("delta").load(direct_bank_source_path)
    ggm_raw = spark.read.format("delta").load(ggm_np_source_path)

    print("[siebel_id_update] stage: normalize sources")
    direct = normalize_active_direct_bank(direct_raw, open_end_ts=open_ended_ts)
    ggm = normalize_active_ggm_np(ggm_raw, open_end_ts=open_ended_ts)
    assert_single_active_seed_per_rel_id(direct, open_ended_ts)
    # Multi-seed input is allowed; deterministic tie-break resolves winners.

    print("[siebel_id_update] stage: build current edges and seeds")
    edges_cur = (
        direct.select(F.lit("direct_bank").alias("source_table"), "rel_id", "id_key")
        .unionByName(ggm.select(F.lit("ggm_np").alias("source_table"), "rel_id", "id_key"))
        .dropDuplicates(["source_table", "rel_id", "id_key"])
)

    seeds_cur = (
        direct.filter((F.col("drc_bnk_f") == "Y") & is_open_ended_ts("edl_valid_to_ts", open_ended_ts))
        .select(F.col("rel_id").alias("seed_rel_id"), F.col("edl_valid_from_ts").alias("seed_valid_from_dts"))
        .dropDuplicates(["seed_rel_id", "seed_valid_from_dts"])
)
    seeds_cur_snapshot = (
        seeds_cur.groupBy("seed_rel_id")
        .agg(F.max("seed_valid_from_dts").alias("seed_valid_from_dts"))
        .dropDuplicates(["seed_rel_id", "seed_valid_from_dts"])
    )

    print("[siebel_id_update] stage: load previous state snapshots")
    edges_prev = spark.table(f"{state_db}.{table_edge_snapshot}").dropDuplicates(["source_table", "rel_id", "id_key"])
    seeds_prev_raw = spark.table(f"{state_db}.{table_seed_snapshot}")
    if "seed_valid_from_dts" in seeds_prev_raw.columns:
        seeds_prev = seeds_prev_raw.select(
            F.col("seed_rel_id").cast("string").alias("seed_rel_id"),
            F.to_timestamp(F.col("seed_valid_from_dts")).alias("seed_valid_from_dts"),
        ).dropDuplicates(["seed_rel_id", "seed_valid_from_dts"])
    else:
        # Backward compatibility with pre-migration snapshots that stored seed_rel_id only.
        seeds_prev = seeds_prev_raw.select(
            F.col("seed_rel_id").cast("string").alias("seed_rel_id"),
            F.lit(None).cast("timestamp").alias("seed_valid_from_dts"),
        ).dropDuplicates(["seed_rel_id", "seed_valid_from_dts"])
    membership_prev = spark.table(f"{state_db}.{table_seed_membership}").dropDuplicates(["rel_id", "rel_id_regie_klant"])

    print("[siebel_id_update] stage: compute deltas")
    edge_cols = ["source_table", "rel_id", "id_key"]
    edge_added, edge_removed = anti_diff(edges_cur, edges_prev, edge_cols)
    edge_changed = edge_added.unionByName(edge_removed).dropDuplicates(edge_cols)

    seed_added, seed_removed = anti_diff(
        seeds_cur_snapshot,
        seeds_prev,
        ["seed_rel_id", "seed_valid_from_dts"],
    )
    seed_added_ids = seed_added.select("seed_rel_id").dropDuplicates(["seed_rel_id"])
    seed_removed_ids = seed_removed.select("seed_rel_id").dropDuplicates(["seed_rel_id"])

    affected_rel_from_removed = (
        membership_prev.join(seed_removed_ids, membership_prev.rel_id_regie_klant == seed_removed_ids.seed_rel_id, "inner")
        .select("rel_id")
        .dropDuplicates(["rel_id"])
)

    impact_start_rel = (
        edge_changed.select("rel_id")
        .unionByName(seed_added_ids.select(F.col("seed_rel_id").alias("rel_id")))
        .unionByName(seed_removed_ids.select(F.col("seed_rel_id").alias("rel_id")))
        .unionByName(affected_rel_from_removed.select("rel_id"))
        .dropDuplicates(["rel_id"])
)

    edges_full = edges_cur.select("rel_id", "id_key").dropDuplicates(["rel_id", "id_key"])
    rel_truncated = False
    print("[siebel_id_update] stage: compute impacted rel_id set")
    if not has_rows(edges_prev) and not has_rows(seeds_prev) and not has_rows(membership_prev):
        impacted_rel_cur = edges_full.select("rel_id").dropDuplicates(["rel_id"])

        print("[siebel_id_update] stage: bootstrap run detected, all current rel_id considered impacted")
    elif not has_rows(impact_start_rel):
        impacted_rel_cur = spark.createDataFrame([], "rel_id string")
        print("[siebel_id_update] stage: no impact detected")
    else:
        impacted_rel_cur, rel_truncated = expand_rel_component(
            impact_start_rel,
            edges_full,
            max_iter=max_iter,
            checkpoint_every=checkpoint_every,
        )

    impacted_seeds_prev = (
        membership_prev.join(impacted_rel_cur, "rel_id", "inner")
        .select(F.col("rel_id_regie_klant").alias("seed_rel_id"))
        .dropDuplicates(["seed_rel_id"])
)

    impacted_seeds = (
        impacted_seeds_prev
        .unionByName(seed_added.select("seed_rel_id"))
        .unionByName(seed_removed_ids.select("seed_rel_id"))
        .dropDuplicates(["seed_rel_id"])
)

    impacted_active_seeds = impacted_seeds.join(
        seeds_cur.select("seed_rel_id").dropDuplicates(["seed_rel_id"]),
        "seed_rel_id",
        "inner",
    )

    impacted_rel_prev = (
        membership_prev
        .join(seed_removed_ids.withColumnRenamed("seed_rel_id", "rel_id_regie_klant"), "rel_id_regie_klant", "inner")
        .select("rel_id")
        .dropDuplicates(["rel_id"])
)

    impacted_rel_all = impacted_rel_cur.unionByName(impacted_rel_prev).dropDuplicates(["rel_id"])


    seed_truncated = False
    print("[siebel_id_update] stage: recompute mappings for impacted graph")
    if not has_rows(impacted_active_seeds):
        recomputed = spark.createDataFrame([], "REL_ID string, REL_ID_REGIE_KLANT string")
        seed_pairs_debug = spark.createDataFrame([], "seed_rel_id string, rel_id string")
        print("[siebel_id_update] stage: no impacted active seeds, writing empty debug table")
        seed_pairs_debug.write.mode("overwrite").format("delta").saveAsTable(debug_table)
        spark.sql(f"CREATE OR REPLACE TEMP VIEW seed_pairs_debug AS SELECT * FROM {debug_table}")
    else:
        seed_pairs, seed_truncated = expand_seed_pairs(
            impacted_active_seeds,
            edges_full,
            max_iter=max_iter,
            checkpoint_every=checkpoint_every,
        )
        seed_pairs = seed_pairs.join(impacted_rel_all, "rel_id", "inner")

        print(f"[siebel_id_update] stage: writing debug seed pairs to {debug_table}")
        seed_pairs.write.mode("overwrite").format("delta").saveAsTable(debug_table)
        spark.sql(f"CREATE OR REPLACE TEMP VIEW seed_pairs_debug AS SELECT * FROM {debug_table}")
        seed_pairs_debug = spark.table(debug_table)

        # Multiple reachable active seeds per rel_id are resolved deterministically.

        if seed_tie_breaker == "latest_then_numeric":
            seed_rank = (
                seeds_cur.groupBy("seed_rel_id")
                .agg(F.max("seed_valid_from_dts").alias("seed_valid_from_dts"))
                .withColumn("seed_num", F.col("seed_rel_id").cast("bigint"))
            )
            recomputed = (
                seed_pairs_debug.join(seed_rank, "seed_rel_id", "left")
                .withColumn(
                    "_rn",
                    F.row_number().over(
                        Window.partitionBy("rel_id").orderBy(
                            F.col("seed_valid_from_dts").desc_nulls_last(),
                            F.col("seed_num").asc_nulls_last(),
                            F.col("seed_rel_id").asc(),
                        )
                    ),
                )
                .filter(F.col("_rn") == 1)
                .select(F.col("rel_id").alias("REL_ID"), F.col("seed_rel_id").alias("REL_ID_REGIE_KLANT"))
            )
        else:
            recomputed = (
                seed_pairs_debug.withColumn("seed_num", F.col("seed_rel_id").cast("bigint"))
                .groupBy("rel_id")
                .agg(
                    F.min(F.when(F.col("seed_num").isNotNull(), F.col("seed_num"))).alias("min_seed_num"),
                    F.min("seed_rel_id").alias("min_seed_lex"),
                )
                .withColumn(
                    "REL_ID_REGIE_KLANT",
                    F.when(F.col("min_seed_num").isNotNull(), F.col("min_seed_num").cast("string")).otherwise(F.col("min_seed_lex")),
                )
                .select(F.col("rel_id").alias("REL_ID"), "REL_ID_REGIE_KLANT")
            )

        recomputed = recomputed.dropDuplicates(["REL_ID"])

    print("[siebel_id_update] stage: validate truncation policy")
    if fail_on_incomplete and (rel_truncated or seed_truncated):
        raise RuntimeError(
            f"Graph expansion truncated (rel_truncated={rel_truncated}, seed_truncated={seed_truncated}) at max_iter={max_iter}"
        )

    print("[siebel_id_update] stage: compute run metrics")
    changed_edges_count = edge_changed.count()
    added_seeds_count = seed_added_ids.count()
    removed_seeds_count = seed_removed_ids.count()
    impacted_rel_count = impacted_rel_all.count()

    output_prev = spark.table(f"{target_db}.{table_output}").select(
        F.col("REL_ID").cast("string").alias("REL_ID"),
        F.col("REL_ID_REGIE_KLANT").cast("string").alias("REL_ID_REGIE_KLANT"),
    )

    print("[siebel_id_update] stage: merge recomputed mappings into output")
    if not has_rows(impacted_rel_all):
        output_new = output_prev
    else:
        old_unimpacted = output_prev.join(
            impacted_rel_all.select(F.col("rel_id").alias("REL_ID")),
            "REL_ID",
            "left_anti",
        )
        output_new = old_unimpacted.unionByName(recomputed).dropDuplicates(["REL_ID"])


    output_new = output_new.join(
        seeds_cur.select(F.col("seed_rel_id").alias("REL_ID_REGIE_KLANT")).dropDuplicates(["REL_ID_REGIE_KLANT"]),
        "REL_ID_REGIE_KLANT",
        "inner",
    )

    print("[siebel_id_update] stage: persist output and state tables")
    overwrite_table_in_place(spark, f"{target_db}.{table_output}", output_new, fmt="delta")
    output_new_local = spark.table(f"{target_db}.{table_output}").select(
        F.col("REL_ID").cast("string").alias("REL_ID"),
        F.col("REL_ID_REGIE_KLANT").cast("string").alias("REL_ID_REGIE_KLANT"),
    )

    overwrite_table_in_place(spark, f"{state_db}.{table_edge_snapshot}", edges_cur, fmt="delta")
    seed_snapshot_table = f"{state_db}.{table_seed_snapshot}"
    seed_snapshot_df = seeds_cur_snapshot.select("seed_rel_id", "seed_valid_from_dts")
    try:
        overwrite_table_in_place(spark, seed_snapshot_table, seed_snapshot_df, fmt="delta")
    except ValueError:
        # One-time migration path from old snapshot schema (seed_rel_id only).
        spark.sql(f"DROP TABLE IF EXISTS {seed_snapshot_table}")
        seed_snapshot_df.write.mode("overwrite").format("delta").saveAsTable(seed_snapshot_table)
    overwrite_table_in_place(
        spark,
        f"{state_db}.{table_seed_membership}",
        output_new_local.select(F.col("REL_ID").alias("rel_id"), F.col("REL_ID_REGIE_KLANT").alias("rel_id_regie_klant")),
        fmt="delta",
    )

    print("[siebel_id_update] stage: append run audit")
    run_id = spark.sql("SELECT uuid() AS run_id").collect()[0]["run_id"]
    audit_row = spark.createDataFrame(
        [
            (
                run_id,
                changed_edges_count,
                added_seeds_count,
                removed_seeds_count,
                impacted_rel_count,
                output_new_local.count(),
                rel_truncated,
                seed_truncated,
            )
        ],
        [
            "run_id",
            "changed_edges",
            "added_seeds",
            "removed_seeds",
            "impacted_rel_ids",
            "output_rows",
            "rel_truncated",
            "seed_truncated",
        ],
    ).withColumn("run_ts", F.current_timestamp()).select(
        "run_id",
        "run_ts",
        "changed_edges",
        "added_seeds",
        "removed_seeds",
        "impacted_rel_ids",
        "output_rows",
        "rel_truncated",
        "seed_truncated",
    )

    run_audit_prev = spark.table(f"{state_db}.{table_run_audit}")
    overwrite_table_in_place(spark, f"{state_db}.{table_run_audit}", run_audit_prev.unionByName(audit_row), fmt="delta")

    spark.catalog.clearCache()

    stats = {
        "changed_edges": changed_edges_count,
        "added_seeds": added_seeds_count,
        "removed_seeds": removed_seeds_count,
        "impacted_rel_ids": impacted_rel_count,
        "output_rows": output_new_local.count(),
        "debug_seed_pairs_table": debug_table,
    }

    if show_samples:
        print("Run stats:", stats)
        output_sample = output_new_local.orderBy("REL_ID").limit(50)
        try:
            display(output_sample)
        except NameError:
            output_sample.show(50, truncate=False)

    print(f"[siebel_id_update] run_incremental_update_databricks: done stats={stats}")
    return stats

In [ ]:
stats = run_incremental_update_databricks(
    spark=spark,
    target_db=target_db,
    state_db=state_db,
    max_iter=max_iter,
    checkpoint_every=checkpoint_every,
    open_ended_ts=open_ended_ts,
    fail_on_incomplete=fail_on_incomplete,
    seed_tie_breaker=seed_tie_breaker,
    show_samples=show_samples,
    seed_membership_source_path=seed_membership_bootstrap_path,
    direct_bank_source_path=direct_bank_source_path,
    ggm_np_source_path=ggm_np_source_path,
)

print("Run stats:", stats)
spark.sql(f"SHOW TABLES IN {target_db}").where(F.col("tableName") == table_output).show(truncate=False)
spark.table(f"{target_db}.{table_output}").orderBy("REL_ID").show(20, truncate=False)